# v5's smoke scene on Colab (scannet/scene0025_01)

The one-scene smoke step of the v5 dev package (`README.md`, "v5 on the dev scenes", step 1; the agent prompt's step
2), on Colab instead of the cluster, then its post-processing:

1. **SAM 3** (`facebook/sam3` at `3c879f3`, transformers backend) on all 128 frames with pipeline_v4's `fixed_v2`
   prompts and synonyms, each detection kept from 0.2, pooled per object (`v5_dev.py`).
2. **Tracking, four ways** on those masks with the revamped tracker (`119e603`; overlap association, mask depth rule,
   tracks from 4 of 128 frames), patched with `--det-score` and per-track scores: v5-his (0.55, `min_points` 50),
   v5-det (0.4, 50), v5-pts (0.55, 300), v5-ours (0.4, 300).
3. **The README's smoke checks**, the same as the cluster run's.
4. **`v5_postprocess.py`** per configuration (point merge, dedup, median scale, trimmed-box relations), and a summary:
   question 3139's rows and relation line, and the scorecard's key box-size measures for the scene.

**Geometry:** the scene's 128 frames, cameras and dense depth converted by the revamped pipeline, taken from Nicolás's
smoke run (`v5_smoke_geometry.zip`; its cameras equal this repository's reconstruction of the scene exactly).
`v5_dev.py` uses a converted scene folder as it is (`geometry_ready`), so no VGGT export is needed here.

About 15–25 minutes on an L4 (installing 5, the SAM 3 weights 2–3, SAM 3 on 12 prompts x 128 frames about 5–8, the
four trackings 3–5, the rest 2); less on an A100.

**Hugging Face access:** `facebook/sam3` is gated. With the account whose token you use, request access on
https://huggingface.co/facebook/sam3 and wait for approval, then add a read token as the Colab secret `HF_TOKEN` (key
icon, notebook access on).

**Files on Google Drive**, in `MyDrive/mvp/`: `mvp-3d-vlm-v5smoke.zip` (the code; its comment is the commit) and
`v5_smoke_geometry.zip`. The results go to `MyDrive/mvp/v5_smoke_results/`.

In [ ]:
DRIVE_DIR = "/content/drive/MyDrive/mvp"
CODE_ZIP = f"{DRIVE_DIR}/mvp-3d-vlm-v5smoke.zip"
GEOMETRY_ZIP = f"{DRIVE_DIR}/v5_smoke_geometry.zip"
RESULTS_DIR = f"{DRIVE_DIR}/v5_smoke_results"
SCENE, QUESTION = "scannet/scene0025_01", 3139
VSI_REPO, VSI_REVISION = "nyu-visionx/VSI-Bench", "3a6a1cf9ac54c403509b15260897cc6ddc41b408"
VSI_SHA256 = "aa3ff8cc2010e555f4f865ff96483812ec0b275005b68562f53bb114ad4db9db"
VGGT_COMMIT = "a288dd0f14786c93483e45524328726ab7b1b4ce"  # OpenVocab-4D's; the tracker imports its geometry helpers

import json, os, shutil, subprocess, sys, zipfile
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
for f in (CODE_ZIP, GEOMETRY_ZIP):
    assert Path(f).is_file(), f"missing {f}: upload it to {DRIVE_DIR}"
CODE, WORK, OUT, SHEETS = (Path("/content/mvp-3d-vlm"), Path("/content/v5_work"), Path("/content/v5_dev_smoke"),
                           Path("/content/v5_sheets"))
for d in (CODE, WORK, OUT, SHEETS):
    shutil.rmtree(d, ignore_errors=True)
with zipfile.ZipFile(CODE_ZIP) as z:
    z.extractall(CODE)
    CODE_COMMIT = z.comment.decode() or "unknown"
E = CODE / "experiments/2026-11-18-scene-sheet-formats"
assert (E / "v5_dev.py").is_file() and "geometry_ready" in (E / "v5_dev.py").read_text(), f"{CODE_ZIP} has no v5_dev.py with geometry_ready"
folder = SCENE.replace("/", "__", 1)
with zipfile.ZipFile(GEOMETRY_ZIP) as z:
    z.extractall(WORK)
SCENE_DIR = WORK / folder / "all" / "scene"
assert (SCENE_DIR / "adapter_report.json").is_file(), f"{GEOMETRY_ZIP} has no {folder}/all/scene/"

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==5.20.1"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps",
                f"git+https://github.com/facebookresearch/vggt@{VGGT_COMMIT}"], check=True)
import torch
assert torch.cuda.is_available(), "no GPU: Runtime > Change runtime type > L4 or A100"
GPU = torch.cuda.get_device_name(0)
from transformers import Sam3Model  # noqa: F401  fails here if this transformers has no SAM 3
ENV = {**os.environ, "PYTHONPATH": str(CODE / "src")}
print(f"code {CODE_COMMIT}; GPU {GPU}; geometry in {SCENE_DIR}")

## Access, data and checks before the GPU work

In [ ]:
import hashlib
import huggingface_hub
from google.colab import userdata

try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception as error:
    raise SystemExit("add a Hugging Face token as the Colab secret HF_TOKEN (key icon, notebook access on)") from error
sys.path.insert(0, str(CODE / "src/mvp/openvocab_adapter"))
import sam3_infer
try:
    huggingface_hub.hf_hub_download("facebook/sam3", "config.json", revision=sam3_infer.SAM3_REV)
except huggingface_hub.errors.GatedRepoError as error:
    raise SystemExit("this token's account has no access to facebook/sam3 yet: request it and wait for approval") from error
ENV["HF_TOKEN"] = os.environ["HF_TOKEN"]
print(f"facebook/sam3@{sam3_infer.SAM3_REV[:7]}: access ok")

# VSI-Bench's questions where v5_dev.py reads them, pinned and checked
raw = CODE / "data/raw/VSI-Bench"
raw.mkdir(parents=True, exist_ok=True)
shutil.copy(huggingface_hub.hf_hub_download(VSI_REPO, "test.jsonl", repo_type="dataset", revision=VSI_REVISION),
            raw / "test.jsonl")
assert hashlib.sha256((raw / "test.jsonl").read_bytes()).hexdigest() == VSI_SHA256, "not the expected test.jsonl"

def run(cmd, **kw):
    r = subprocess.run(cmd, cwd=CODE, env=ENV, capture_output=True, text=True, **kw)
    print(r.stdout[-3000:])
    assert r.returncode == 0, r.stderr[-4000:]
    return r.stdout

# the scene's gold sheets (for v5_postprocess.py's index and the comparison), from the scene's questions only
scene_q = Path("/content/scene_questions.jsonl")
scene_q.write_text("".join(l + "\n" for l in (raw / "test.jsonl").read_text().splitlines()
                           if json.loads(l)["dataset"] + "/" + json.loads(l)["scene_name"] == SCENE))
GOLD = CODE / "tasks/openvocab_adapter/out_gold_scene"
run([sys.executable, str(CODE / "experiments/2026-11-19-gold-sheet-relations/build_gold_sheets.py"), "--questions",
     str(scene_q), "--prompts", str(CODE / "experiments/2026-11-19-gold-sheet-relations/prompts_full.jsonl"),
     "--out", str(GOLD)])

# the README's step: test_v5.py (without .git the tracker-patch tests are skipped; the run itself patches the tracker)
r = subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", str(E), "-p", "test_v5.py"], cwd=CODE, env=ENV,
                   capture_output=True, text=True)
print(r.stderr[-600:])
assert r.returncode == 0, "test_v5.py failed: stop here"

## SAM 3 and the four trackings (`v5_dev.py`)

The same command as the cluster's smoke step, with this scene's converted geometry already in `--work`. It writes
progress to `v5_dev.log` in `--out`.

In [ ]:
import time

scenes_file = Path("/content/v5_smoke_scene.txt")
scenes_file.write_text(SCENE + "\n")
t0 = time.time()
proc = subprocess.Popen([sys.executable, "-u", str(E / "v5_dev.py"), "--scenes", str(scenes_file), "--work", str(WORK),
                         "--out", str(OUT), "--data-root", "/content/no_export_needed"],
                        cwd=CODE, env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
lines = []
for line in proc.stdout:
    lines.append(line)
    print(line, end="")
code_ = proc.wait()
print(f"v5_dev.py: exit {code_}, {(time.time() - t0) / 60:.1f} min on {GPU}")
if code_ != 0:
    print((OUT / "v5_dev.log").read_text()[-5000:] if (OUT / "v5_dev.log").is_file() else "(no log)")
    raise SystemExit("v5_dev.py failed: stop here")

## The README's smoke checks

In [ ]:
sys.path[:0] = [str(CODE / "src"), str(E)]  # v5_dev, and pipeline_scorecard, which imports mvp
import v5_dev

problems = []
finished = [l for l in lines if l.startswith("finished:")]
expected = "finished: " + str({c: 1 for c in v5_dev.CONFIGS})
if not finished or not finished[-1].startswith(expected):
    problems.append(f"last line {finished[-1].strip() if finished else '(none)'!r}, not {expected!r}...")
manifest = json.loads((WORK / folder / "all/masks/sam3_raw/manifest.json").read_text())
if len(manifest["frames"]) != 128 or manifest["min_score"] != 0.2:
    problems.append(f"SAM 3 manifest: {len(manifest['frames'])} frames, min_score {manifest['min_score']}")
n_questions = sum(1 for _ in scene_q.read_text().splitlines())
for config, (det, min_points) in v5_dev.CONFIGS.items():
    base = OUT / config / folder / "all"
    if not (base / "DONE.json").is_file():
        problems.append(f"{config}: no DONE.json")
        continue
    done = json.loads((base / "DONE.json").read_text())
    summary = json.loads((base / "scene/tracks/summary.json").read_text())
    if summary.get("association", {}).get("rule") != "overlap" or summary.get("min_track_frames") != 4 \
            or summary.get("det_score") != det or summary.get("depth_rule", {}).get("min_points") != min_points:
        problems.append(f"{config}: summary.json {summary.get('association', {}).get('rule')}, min_track_frames "
                        f"{summary.get('min_track_frames')}, det_score {summary.get('det_score')}, min_points "
                        f"{summary.get('depth_rule', {}).get('min_points')}")
    names = list(done["names"])
    no_scores = [n for n in names if not (base / "scene/tracks" / f"{n.replace(' ', '_')}.scores.json").is_file()]
    if no_scores:
        problems.append(f"{config}: no scores.json for {no_scores}")
    sheets = list((base / "sheets").glob("q*/question.json"))
    if len(sheets) != n_questions:
        problems.append(f"{config}: {len(sheets)} question folders, not {n_questions}")
    objects = {n: summary[n]["objects"] for n in names if n in summary}
    print(f"{config}: det {det}, min_points {min_points}; tracked {objects}; "
          f"SAM 3 {done['sam3']} s, tracking {done['tracking_seconds']} s")
print(f"SAM 3: {len(manifest['prompts'])} prompts x {len(manifest['frames'])} frames, from {manifest['min_score']}, "
      f"{manifest['seconds']} s on {manifest['gpu']}")
assert not problems, "the smoke checks failed:\n  " + "\n  ".join(problems)
print(f"OK: every smoke check passes ({n_questions} questions, {len(names)} object names)")

## Post-processing and the summary (`v5_postprocess.py`)

The run leaves out the depth; the post-processing takes the scene's `depth.npz` the tracker used (the same file, in
`--work`) instead of making it again from `data/vggt`, which is not here.

In [ ]:
results = {}
for config in v5_dev.CONFIGS:
    scene_out = OUT / config / folder / "all/scene"
    shutil.copy(SCENE_DIR / "depth.npz", scene_out / "depth.npz")
    out = run([sys.executable, str(E / "v5_postprocess.py"), "--run", str(OUT / config), "--questions",
               str(raw / "test.jsonl"), "--gold-root", str(GOLD), "--out", str(SHEETS / config)])
    results[config] = json.loads(out)
    print(config, "scale", {k: round(v, 3) if v else v for k, v in results[config]["scales"][SCENE].items()})

In [ ]:
import csv
from collections import Counter
import numpy as np
import pipeline_scorecard as sc

def read(path):
    with open(path, newline="") as f:
        return list(csv.DictReader(f))

index = {config: {int(r["id"]): r for r in read(SHEETS / config / "index.csv")} for config in v5_dev.CONFIGS}
gold_rows = read(SHEETS / "v5-his" / index["v5-his"][QUESTION]["gold_sheet"] / "objects.csv")
print(f"question {QUESTION}: {index['v5-his'][QUESTION]['question']}")
print(f"gold rows: {dict(Counter(r['concept'] for r in gold_rows))}; ground truth {index['v5-his'][QUESTION]['ground_truth']}\n")
table = []
for config in v5_dev.CONFIGS:
    rows = read(SHEETS / config / index[config][QUESTION]["pipeline_sheet"] / "objects.csv")
    rel = json.loads((SHEETS / config / "relations_v5.json").read_text()).get(str(QUESTION), "")
    line = [l for l in rel.splitlines() if l.startswith("Distance from each candidate")]
    rec = sc.score_root(SHEETS / config, {SCENE})[0]
    table.append({"config": config, "scale": round(results[config]["scales"][SCENE]["scale"], 3),
                  "rows (3139)": dict(Counter(r["concept"] for r in rows)),
                  "relation line (3139)": line[0] if line else "(none)",
                  "box side / gold, median": round(float(np.median(rec["box_ratio"])), 2) if rec["box_ratio"] else None,
                  "boxes over 1.5x": f"{100 * np.mean(rec['box_inflated']):.0f}%" if rec["box_inflated"] else None,
                  "gap error, median (m)": round(float(np.median(rec["gap_abs"])), 2) if rec["gap_abs"] else None,
                  "named objects found": f"{100 * np.mean(rec['found']):.0f}%",
                  "count = gold": f"{100 * np.mean(rec['count_match']):.0f}%"})
for t in table:
    print(json.dumps(t))

## Save to Drive

In [ ]:
Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)
for config in v5_dev.CONFIGS:
    (OUT / config / folder / "all/scene/depth.npz").unlink(missing_ok=True)
for name, src in (("v5_dev_smoke", OUT), ("v5_sheets", SHEETS)):
    shutil.make_archive(f"{RESULTS_DIR}/{name}", "zip", src.parent, src.name)
shutil.copy(WORK / folder / "all/masks/sam3_raw/manifest.json", f"{RESULTS_DIR}/sam3_manifest.json")
Path(f"{RESULTS_DIR}/summary.json").write_text(json.dumps(
    {"code_commit": CODE_COMMIT, "gpu": GPU, "scene": SCENE, "question": QUESTION, "configs": table,
     "postprocess": results}, indent=1))
print("saved:", sorted(p.name for p in Path(RESULTS_DIR).iterdir()))